# Part 1: Neural Network Fundamentals and Training Behavior Analysis
**Dataset:** Customer Churn Dataset (customer_churn_nn.csv)  
**Problem Type:** Binary Classification — Churn Prediction  
**Dataset Source:** https://drive.google.com/drive/folders/1akV6po4Nrgkc3yQrJkzA6cJlV-wBvUYs?usp=sharing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

os.makedirs('results', exist_ok=True)
print('Libraries loaded.')
print(f'TensorFlow: {tf.__version__}')

## Task 1: Dataset Understanding

In [ ]:
# Synthetic customer churn dataset
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    'customer_id': range(1, n+1),
    'age': np.random.randint(18, 70, n),
    'tenure': np.random.randint(1, 72, n),
    'monthly_charges': np.round(np.random.uniform(20, 120, n), 2),
    'total_charges': np.round(np.random.uniform(100, 8000, n), 2),
    'contract_type': np.random.choice(['Month-to-Month', 'One Year', 'Two Year'], n, p=[0.5, 0.3, 0.2]),
    'payment_method': np.random.choice(['Credit Card', 'Bank Transfer', 'Electronic Check', 'Mailed Check'], n),
    'support_calls': np.random.randint(0, 10, n),
    'satisfaction_score': np.random.randint(1, 6, n),
    'product_subscriptions': np.random.randint(1, 5, n),
})

# Churn label: influenced by contract type, satisfaction, and support calls
churn_prob = (
    (df['contract_type'] == 'Month-to-Month').astype(float) * 0.4 +
    (df['satisfaction_score'] <= 2).astype(float) * 0.3 +
    (df['support_calls'] >= 7).astype(float) * 0.2 +
    np.random.uniform(0, 0.1, n)
)
df['churn'] = (churn_prob > 0.45).astype(int)

print(f'Dataset shape: {df.shape}')
print(f'\nChurn distribution:\n{df["churn"].value_counts()}')
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'\nData types:\n{df.dtypes}')
df.head()

## Task 2: Data Preprocessing

In [ ]:
# Drop non-informative column
df = df.drop('customer_id', axis=1)

# Label encode categoricals
le = LabelEncoder()
df['contract_type'] = le.fit_transform(df['contract_type'])
df['payment_method'] = le.fit_transform(df['payment_method'])

# Split features and target
X = df.drop('churn', axis=1)
y = df['churn']

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {X_train_scaled.shape[0]}')
print(f'Test samples: {X_test_scaled.shape[0]}')
print(f'Features: {X_train_scaled.shape[1]}')

## Task 3: Neural Network Architecture

In [ ]:
def build_model(hidden_units=64, hidden_layers=2, activation='relu', learning_rate=0.001):
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train_scaled.shape[1],)))
    for _ in range(hidden_layers):
        model.add(layers.Dense(hidden_units, activation=activation))
        model.add(layers.Dropout(0.2))
    model.add(layers.Dense(1, activation='sigmoid'))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Build baseline model
baseline_model = build_model()
baseline_model.summary()

## Task 4: Training and Evaluation

In [ ]:
# Train baseline model
history = baseline_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=50, batch_size=32, verbose=0
)

# Evaluate
y_pred = (baseline_model.predict(X_test_scaled) > 0.5).astype(int).flatten()
test_acc = accuracy_score(y_test, y_pred)
print(f'Baseline Test Accuracy: {test_acc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Churn','Churn'], yticklabels=['No Churn','Churn'])
plt.title('Confusion Matrix — Baseline NN')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=150)
plt.show()

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Val')
axes[1].set_title('Accuracy'); axes[1].legend()
plt.tight_layout()
plt.savefig('results/evaluation_outputs.png', dpi=150)
plt.show()

## Task 5: Hyperparameter Experiments

In [ ]:
experiments = [
    {'name': 'Baseline', 'hidden_units': 64, 'hidden_layers': 2, 'activation': 'relu', 'learning_rate': 0.001, 'batch_size': 32},
    {'name': 'More Neurons (128)', 'hidden_units': 128, 'hidden_layers': 2, 'activation': 'relu', 'learning_rate': 0.001, 'batch_size': 32},
    {'name': 'Deeper Network (3L)', 'hidden_units': 64, 'hidden_layers': 3, 'activation': 'relu', 'learning_rate': 0.001, 'batch_size': 32},
    {'name': 'Lower LR (0.0001)', 'hidden_units': 64, 'hidden_layers': 2, 'activation': 'relu', 'learning_rate': 0.0001, 'batch_size': 32},
    {'name': 'Tanh Activation', 'hidden_units': 64, 'hidden_layers': 2, 'activation': 'tanh', 'learning_rate': 0.001, 'batch_size': 32},
    {'name': 'Larger Batch (128)', 'hidden_units': 64, 'hidden_layers': 2, 'activation': 'relu', 'learning_rate': 0.001, 'batch_size': 128},
]

results = []
for exp in experiments:
    m = build_model(exp['hidden_units'], exp['hidden_layers'], exp['activation'], exp['learning_rate'])
    h = m.fit(X_train_scaled, y_train, epochs=30, batch_size=exp['batch_size'], verbose=0)
    tr_acc = h.history['accuracy'][-1]
    y_p = (m.predict(X_test_scaled, verbose=0) > 0.5).astype(int).flatten()
    te_acc = accuracy_score(y_test, y_p)
    results.append({'Experiment': exp['name'], 'Train Acc': round(tr_acc, 4), 'Test Acc': round(te_acc, 4)})
    print(f"{exp['name']:30s} Train: {tr_acc:.4f}  Test: {te_acc:.4f}")

comp_df = pd.DataFrame(results)
comp_df.to_csv('results/model_comparison_table.csv', index=False)
print('\nAll experiments complete.')
comp_df

## Task 6: Reflection

### Weights and Biases
Weights and biases are the learnable parameters of a neural network. During training, they are updated via **backpropagation** — the gradient of the loss function is computed with respect to each parameter, and the optimizer (Adam) adjusts them to minimize the loss.

### Role of Activation Functions
Activation functions introduce **non-linearity** into the network. Without them, a deep neural network collapses into a linear model regardless of depth. ReLU is preferred for hidden layers because it avoids the vanishing gradient problem. Sigmoid is used in the output layer for binary classification to squash outputs to [0, 1].

### Effect of Learning Rate
- **Too high (e.g., 0.1):** Loss oscillates or diverges — the optimizer overshoots the minimum
- **Too low (e.g., 0.0001):** Convergence is slow — the model needs many more epochs
- **Optimal (e.g., 0.001):** Steady convergence with good generalization

### Overfitting and Underfitting
- **Overfitting:** Train accuracy >> Test accuracy (model memorizes training data)
- **Underfitting:** Both train and test accuracy are low (model too simple)
- **Mitigation:** Dropout regularization, early stopping, cross-validation